<h2>Semester project intended for subject IAU</h2>

<h3>
The main objective is to predict oxygen saturation dependent variable using all available features and information about independent variables and the dataset itself
</h3>

In [ ]:
import re
import pandas as pd
import matplotlib as mpl
import numpy as np

In [2]:
observations = pd.read_csv("Datasets/observation.csv", sep = '\t')
#patients file is corrupted and needs to be reformated
stations = pd.read_csv("Datasets/station.csv", sep = '\t')

In [3]:
stations.columns

Index(['QoS', 'longitude', 'station', 'latitude', 'code', 'revision'], dtype='object')

In [4]:
stations.tail(10)

,QoS,longitude,station,latitude,code,revision
734,good,-77.96389,Martinsburg,39.45621,US,"02/10/2024, 00:00:00"
735,good,-87.71255,South Lawndale,41.84364,US,2017/10/03
736,good,8.51667,Verl,51.88333,DE,2020/08/09
737,good,120.74221,Changshu City,31.64615,CN,28 Mar 2016
738,excellent,72.93924,Uran,18.87813,IN,26 Feb 2024
739,good,9.22796,Pfullingen,48.46458,DE,2017-07-28
740,good,-76.99081,Chillum,38.96372,US,2023-07-10
741,excellent,-80.22588,North Lauderdale,26.21730,US,"07/29/2024, 00:00:00"
742,good,122.08333,Ulanhot,46.08333,CN,03 Nov 2020
743,excellent,51.34520,Otradnyy,53.37596,RU,2022-05-31


In [5]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   QoS        744 non-null    object 
 1   longitude  744 non-null    float64
 2   station    744 non-null    object 
 3   latitude   744 non-null    float64
 4   code       744 non-null    object 
 5   revision   744 non-null    object 
dtypes: float64(2), object(4)
memory usage: 35.0+ KB


QoS - Quality of Service (Ordinal categorical data type)
<br>
Revision - revision date (Time data type)
<br>
Longtitude - coordinate on earth (Floating point number data type)
<br>
Latitude - coordinate on earth (Floating point number data type)
<br>
Code - country code (e.g US) (Nominal categorical data type)
<br>
Station - station name (Nominal categorical data type)

In [6]:
stations.drop(columns = ["code", "station"], axis = 1, inplace = True)# No need for nominal categorical type of data, additionally if there's any correlation 
# or even causation it would be with longtitude and latitude, both of which represent station and code features


In [7]:
stations.QoS.unique()

array(['good', 'maintenance', 'excellent', 'acceptable'], dtype=object)

Ordinal data points in QoS feature can be substituted for rank numbers to render them more compatible for working with other features

In [8]:
observations.columns

Index(['SpO₂', 'HR', 'PI', 'RR', 'EtCO₂', 'FiO₂', 'PRV', 'BP',
       'Skin Temperature', 'Motion/Activity index', 'PVI', 'Hb level', 'SV',
       'CO', 'Blood Flow Index', 'PPG waveform features',
       'Signal Quality Index', 'Respiratory effort', 'O₂ extraction ratio',
       'SNR', 'oximetry', 'latitude', 'longitude'],
      dtype='object')

In [9]:
observations.tail(10)

,SpO₂,HR,PI,RR,EtCO₂,FiO₂,PRV,BP,Skin Temperature,Motion/Activity index,...,CO,Blood Flow Index,PPG waveform features,Signal Quality Index,Respiratory effort,O₂ extraction ratio,SNR,oximetry,latitude,longitude
12125,96.120592,85.137741,9.818648,16.517384,39.497338,63.465126,108.452385,99.922432,36.607412,9.461546,...,4.179417,67.246736,45.970792,60.964147,46.949956,0.244793,29.972377,1.0,51.39323,0.47713
12126,96.408529,72.257124,13.482344,15.602615,41.102186,59.800858,107.352620,102.193050,36.370945,10.254790,...,4.010835,66.918832,67.578778,55.308383,53.064130,0.217349,35.155641,0.0,45.53929,-122.38731
12127,96.669288,73.117505,6.290254,14.997423,39.577774,64.047982,100.071172,103.295957,36.010458,12.152013,...,4.013708,23.594854,59.034074,57.341219,48.095680,0.269824,34.980192,0.0,14.93333,-91.11667
12128,98.216324,79.663340,11.051551,17.375844,41.078261,44.443769,110.874503,103.165378,34.659919,10.647416,...,4.057127,58.158724,30.875445,52.498185,46.531911,0.287695,28.941719,0.0,25.87972,-97.50417
12129,96.341803,79.506197,12.135653,15.223063,39.617252,49.648834,83.516401,107.477905,34.865892,10.222127,...,4.054676,68.146147,57.681426,45.756827,38.005644,0.223517,39.005482,0.0,10.15031,-73.96140
12130,97.385205,78.466039,13.349946,16.965079,39.771876,65.804506,125.231169,101.359033,35.925456,11.381142,...,4.043600,63.955049,62.704268,54.052104,36.885416,0.215833,37.802841,0.0,14.08230,98.19151
12131,96.930364,65.658406,11.212824,16.809527,38.855673,54.344864,127.384993,96.997014,36.260074,11.105850,...,4.002268,42.773740,58.200613,42.130057,51.738050,0.291330,21.791978,1.0,33.41012,-91.06177
12132,97.150812,72.491221,11.295314,17.996657,41.619844,52.582024,105.552984,110.866776,36.320781,9.961450,...,4.012385,41.823153,33.807926,54.432173,53.776058,0.210734,24.210105,1.0,-10.33333,39.28333
12133,96.715396,77.256318,15.914116,17.622823,39.353100,62.014037,116.798968,101.787271,36.629434,9.621106,...,4.033931,55.404543,20.146844,54.464593,29.602455,0.248375,20.018617,1.0,46.13510,-60.18310
12134,97.677693,82.205473,6.429286,18.149222,39.693213,53.818497,114.064804,106.760584,37.004959,11.230760,...,4.096676,37.290319,50.057570,26.471225,59.439338,0.245908,37.014194,1.0,53.16167,6.76111


In [10]:
observations.columns

Index(['SpO₂', 'HR', 'PI', 'RR', 'EtCO₂', 'FiO₂', 'PRV', 'BP',
       'Skin Temperature', 'Motion/Activity index', 'PVI', 'Hb level', 'SV',
       'CO', 'Blood Flow Index', 'PPG waveform features',
       'Signal Quality Index', 'Respiratory effort', 'O₂ extraction ratio',
       'SNR', 'oximetry', 'latitude', 'longitude'],
      dtype='object')

In [16]:
observations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12135 entries, 0 to 12134
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SpO₂                   12135 non-null  float64
 1   HR                     12135 non-null  float64
 2   PI                     12135 non-null  float64
 3   RR                     12135 non-null  float64
 4   EtCO₂                  12135 non-null  float64
 5   FiO₂                   12135 non-null  float64
 6   PRV                    12135 non-null  float64
 7   BP                     12135 non-null  float64
 8   Skin Temperature       12135 non-null  float64
 9   Motion/Activity index  12135 non-null  float64
 10  PVI                    12135 non-null  float64
 11  Hb level               12135 non-null  float64
 12  SV                     12135 non-null  float64
 13  CO                     12135 non-null  float64
 14  Blood Flow Index       12135 non-null  float64
 15  PP

Latitude and longtitude would be used to join observations and and stations, since it observations logtitude and latitude represent location of the station where each observation was conducted. 
<br>
Later on patients can be joined with the resulting dataframe on station_ID feature.

In [ ]:
file = open("Datasets/patient.csv", "r", encoding="utf-8")
parsed_file = open("Datasets/patient_redacted.csv", "+a")

columns = ["registration", "latitude", "longtitude", "birthdate", "ssn", "mail", "blood_group", "station_ID"]
[parsed_file.write(col + "   ") for col in columns if (columns.index(col) + 1 != len(columns))]
parsed_file.write(columns[-1] + "\n")
new_lines = list()
lines = file.readlines()



i = 0
line_ctr = -1
line_number = len(lines)
while(i < line_number):
    if(not lines[i][0].isspace()):
        new_lines.insert(line_ctr, new_lines[line_ctr] + " " + lines[i].strip())
    else:
        line_ctr += 1
        new_lines.append(lines[i].strip())
        
    i += 1
del lines, i, line_number

for index in range(line_ctr):
    nc_line = new_lines[index]
    location = re.findall("\(Decimal\('.*'\),\sDecimal\('.*'\)\)", nc_line)
    loc_pos = re.search("\(Decimal\('.*'\),\sDecimal\('.*'\)\)", nc_line)
    dates = re.findall("((\d{4}[./-]\d{2}[./-]\d{2})|(\d{2} \w{3} \d{4})|((\d{2}/){2}\d{4}))(,\s(\d{2}:){2}\d{2})?", nc_line)
    date_pos = re.search(f"{dates[0]}", nc_line)
    ssn = re.findall("([A-Z0-9]{16})|(\d{3}-\d{2}-\d{4})|(\d{11})", nc_line)
    ssn_pos = re.search("([A-Z0-9]{16})|(\d{3}-\d{2}-\d{4})|(\d{11})", nc_line)
    mail = re.findall("[\w\d.]*@[\w\d.]*", nc_line)
    blood_group = re.findall("[ABO]{1,2}[+-]", nc_line)
    station_ID = re.findall("\d{1,3}\n", nc_line)
    
    
    location = None if(loc_pos == None) else location = re.match("\(Decimal\('(?P<latitude>.*)'\),\sDecimal\('(?P<longtitude>.*)'\)\)", location[0])
    ssn = None if(ssn_pos == None) else ssn = ssn[0]
    mail = None if(len(mail) == 0) else mail = mail[0]
    blood_group = None if(len(blood_group) == 0) else blood_group = blood_group[0]
    station_ID = None if(len(station_ID) == 0) else station_ID = station_ID[0].strip()
    
    print(f"{len(dates)} - number of dates in a string")
    if(len(dates) > 1):
        registration = dates[0]
        birthdate = dates[1]
    elif(date_pos != None):
        if(re.match("((\d{4}/\d{2}/\d{2})|(\d{2} \w{3} \d{4})|((\d{2}/){2}\d{4}))(,\s(\d{2}:){2}\d{2})?", dates[0]) != None):
            registration = dates[0]
            birthdate = None
        elif(location != None):
            if(date_pos < loc_pos):
                registration = dates[0]
                birthdate = None
            else:
                birthdate = dates[0]
                registration = None
        # elif(ssn_pos != None):
        #     if(ssn_pos - date_pos < 18):
        #         birthdate = dates[0]
        #         registration = None
        # elif(re.search(f"{mail[0]}", nc_line) - date_pos > 30):
        #     registration = dates[0]
        #     birthdate = None
    else:
        registration = None
        birthdate = None
        
    parsed_file.write(f"{registration}  {location["latitude"]}  {location["longtitude"]}    {birthdate} {ssn}   {mail}  {blood_group}   {station_ID}\n")
    
    

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3701111121.py, line 39)